# CUDA Concept and Programming
## Group 3:

Members:
*   Alcasid, Aizy
*   Bilanes, Kendrick
*   Corpuz, Gerald
*   Roa, Luis







CUDA Version 2: No Prefetch, No Mem Advise

In [7]:
%%writefile axpy_cuda_gridstride.cu
#define _POSIX_C_SOURCE 199309L
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>
#include <cuda_runtime.h>

#define NUM_RUNS       30
#define BLOCKS_PER_SM  32

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err__ = (call); \
        if (err__ != cudaSuccess) { \
            fprintf(stderr, "CUDA error at %s:%d: %s\n", \
                    __FILE__, __LINE__, cudaGetErrorString(err__)); \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

void axpy_c(int n, float a, const float* X, const float* Y, float* Z) {
    int i;
    for (i = 0; i < n; i++) {
        Z[i] = a * X[i] + Y[i];
    }
}

__global__ void axpy_gridstride(int n, float a, const float* X, const float* Y, float* Z) {
    int idx    = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = idx; i < n; i += stride) {
        Z[i] = __fadd_rn(__fmul_rn(a, X[i]), Y[i]);
    }
}

void compare(const float ref[], const float target[], int size) {
    int i;
    int ctr = 0;
    float max_diff = 0.0f;
    for (i = 0; i < size; i++) {
        if (ref[i] != target[i]) {
            ctr++;
        }
        float d = fabsf(ref[i] - target[i]);
        if (d > max_diff) max_diff = d;
    }
    printf("Error Count = %d (max |diff| = %.3e)\n", ctr, max_diff);
}

void print_samples(const char* kernel_name, const float Z[], int n) {
    printf("%s Elements:\n  First 5: %.4f %.4f %.4f %.4f %.4f\n",
        kernel_name, Z[0], Z[1], Z[2], Z[3], Z[4]);
    printf("  Last 5:  %.4f %.4f %.4f %.4f %.4f\n",
        Z[n - 5], Z[n - 4], Z[n - 3], Z[n - 2], Z[n - 1]);
}

void run_test(int n, float a, int numSMs) {
    printf("\n======================================================================\n");
    printf(" Testing Vector Size n = %d (Scalar a = %.2f)\n", n, a);
    printf("======================================================================\n");

    int i, run;
    size_t bytes = (size_t)n * sizeof(float);

    float* X = NULL;
    float* Y = NULL;
    float* Z_cuda = NULL;
    float* Z_c = (float*)malloc(bytes);

    cudaError_t e1 = cudaMallocManaged(&X, bytes);
    cudaError_t e2 = cudaMallocManaged(&Y, bytes);
    cudaError_t e3 = cudaMallocManaged(&Z_cuda, bytes);

    if (!Z_c || e1 != cudaSuccess || e2 != cudaSuccess || e3 != cudaSuccess) {
        printf("Memory allocation failed!\n");
        cudaGetLastError();
        cudaFree(X); cudaFree(Y); cudaFree(Z_cuda); free(Z_c);
        return;
    }

    for (i = 0; i < n; i++) {
        X[i] = sinf(i * 0.001f);
        Y[i] = cosf(i * 0.002f);
        Z_c[i] = 0.0f;
        Z_cuda[i] = 0.0f;
    }

    axpy_c(n, a, X, Y, Z_c);

    // Block sizes to benchmark
    int block_sizes[] = {256, 512, 1024};
    int num_block_tests = sizeof(block_sizes) / sizeof(block_sizes[0]);

    for (int b_idx = 0; b_idx < num_block_tests; b_idx++) {
        int block = block_sizes[b_idx];
        int grid  = (n + block - 1) / block;
        int maxGrid = numSMs * BLOCKS_PER_SM;
        if (grid > maxGrid) grid = maxGrid;

        double times[NUM_RUNS];
        for (run = 0; run < NUM_RUNS; run++) {
            struct timespec start, end;
            clock_gettime(CLOCK_MONOTONIC, &start);
            axpy_gridstride<<<grid, block>>>(n, a, X, Y, Z_cuda);
            cudaError_t sync_err = cudaDeviceSynchronize();
            clock_gettime(CLOCK_MONOTONIC, &end);

            CUDA_CHECK(cudaGetLastError());
            CUDA_CHECK(sync_err);

            times[run] = ((end.tv_sec - start.tv_sec) * 1000.0) + ((end.tv_nsec - start.tv_nsec) / 1000000.0);
        }

        double sum_all = 0.0, sum_warm = 0.0;
        double tmin = times[0], tmax = times[0];
        for (run = 0; run < NUM_RUNS; run++) {
            sum_all += times[run];
            if (run > 0) sum_warm += times[run];
            if (times[run] < tmin) tmin = times[run];
            if (times[run] > tmax) tmax = times[run];
        }
        double avg_all  = sum_all / NUM_RUNS;
        double avg_warm = sum_warm / (NUM_RUNS - 1);

        printf("\n--- Block Size: %d Threads ---\n", block);
        printf("Launch config            = %d blocks x %d threads\n", grid, block);
        printf("Time = %.6f ms   (average of all %d runs)\n", avg_all, NUM_RUNS);
        printf("Run 1 (cold)             = %.6f ms\n", times[0]);
        printf("Runs 2-%d avg (warm)     = %.6f ms\n", NUM_RUNS, avg_warm);
        printf("Min / Max                = %.6f / %.6f ms\n", tmin, tmax);
    }

    print_samples("CUDA Vector Z", Z_cuda, n);
    print_samples("C Reference Z", Z_c, n);
    compare(Z_c, Z_cuda, n);

    cudaFree(X);
    cudaFree(Y);
    cudaFree(Z_cuda);
    free(Z_c);
}

int main(int argc, char* argv[]) {
    float a = 2.0f;

    CUDA_CHECK(cudaFree(0));

    int dev = 0;
    CUDA_CHECK(cudaGetDevice(&dev));
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, dev));
    int numSMs = 0, concurrent = 0, managed = 0;
    CUDA_CHECK(cudaDeviceGetAttribute(&numSMs, cudaDevAttrMultiProcessorCount, dev));
    CUDA_CHECK(cudaDeviceGetAttribute(&concurrent, cudaDevAttrConcurrentManagedAccess, dev));
    CUDA_CHECK(cudaDeviceGetAttribute(&managed, cudaDevAttrManagedMemory, dev));

    printf("GPU: %s (%d SMs)\n", prop.name, numSMs);
    printf("managedMemory = %d, concurrentManagedAccess = %d\n", managed, concurrent);
    printf("Data init: X[i] = sinf(i*0.001f), Y[i] = cosf(i*0.002f), Z[i] = 0, a = %.1f\n", a);

    if (argc > 1) {
        int n = atoi(argv[1]);
        if (n < 5) {
            printf("n must be at least 5.\n");
            return 1;
        }
        run_test(n, a, numSMs);
    }
    else {
        int test_sizes[] = {
            1 << 20,
            1 << 24,
            1 << 26,
            1 << 28,
            1003,
            (1 << 20) + 7
        };

        int num_tests = sizeof(test_sizes) / sizeof(test_sizes[0]);

        for (int t = 0; t < num_tests; t++) {
            run_test(test_sizes[t], a, numSMs);
        }
    }

    return 0;
}

Overwriting axpy_cuda_gridstride.cu


In [8]:

!nvcc -O2 -arch=sm_75 axpy_cuda_gridstride.cu -o axpy_cuda_gridstride

!nsys profile --trace=cuda,osrt --output=/content/report_refined --force-overwrite=true --stats=false ./axpy_cuda_gridstride

!nsys stats --force-export=true --report cuda_gpu_sum,cuda_api_sum,cuda_gpu_mem_size_sum /content/report_refined.nsys-rep

axpy_cuda_gridstride.cu:1: warning: "_POSIX_C_SOURCE" redefined
    1 | #define _POSIX_C_SOURCE 199309L
      | 
In file included from /usr/local/cuda/bin/../targets/x86_64-linux/include/crt/host_config.h:208,
                 from /usr/local/cuda/bin/../targets/x86_64-linux/include/cuda_runtime.h:82,
                 from <command-line>:
/usr/include/features.h:292: note: this is the location of the previous definition
  292 | # define _POSIX_C_SOURCE        200809L
      | 
axpy_cuda_gridstride.cu:1: warning: "_POSIX_C_SOURCE" redefined
    1 | #define _POSIX_C_SOURCE 199309L
      | 
In file included from /usr/local/cuda/bin/../targets/x86_64-linux/include/crt/host_config.h:208,
                 from /usr/local/cuda/bin/../targets/x86_64-linux/include/cuda_runtime.h:82,
                 from <command-line>:
/usr/include/features.h:292: note: this is the location of the previous definition
  292 | # define _POSIX_C_SOURCE        200809L
      | 
GPU: Tesla T4 (40 SMs)
managedMemory =

In [9]:
!ncu --set basic -k axpy -c 1 ./axpy_cuda_gridstride 1024

==PROF== Connected to process 14926 (/content/axpy_cuda_gridstride)
GPU: Tesla T4 (40 SMs)
managedMemory = 1, concurrentManagedAccess = 1
Data init: X[i] = sinf(i*0.001f), Y[i] = cosf(i*0.002f), Z[i] = 0, a = 2.0

 Testing Vector Size n = 1024 (Scalar a = 2.00)

--- Block Size: 256 Threads ---
Launch config            = 4 blocks x 256 threads
Time = 0.266918 ms   (average of all 30 runs)
Run 1 (cold)             = 1.350576 ms
Runs 2-30 avg (warm)     = 0.229551 ms
Min / Max                = 0.133523 / 1.350576 ms

--- Block Size: 512 Threads ---
Launch config            = 2 blocks x 512 threads
Time = 0.207276 ms   (average of all 30 runs)
Run 1 (cold)             = 0.296990 ms
Runs 2-30 avg (warm)     = 0.204183 ms
Min / Max                = 0.133885 / 0.859933 ms

--- Block Size: 1024 Threads ---
Launch config            = 1 blocks x 1024 threads
Time = 0.203642 ms   (average of all 30 runs)
Run 1 (cold)             = 0.306197 ms
Runs 2-30 avg (warm)     = 0.200106 ms
Min / Max      